# Adding AgentCore Memory

In this lab, you will add Amazon Bedrock Agentcore Memory as the conversation manager for the Strands Agent, which will give both short term memory for the agent, as well as RAG based personalization for users of the agent.

We will demonstrate how Strands will use the Memory with just a small code change and then how to configure the Long Term memory for personalization.

## Prerequisites
- Complete Lab 3.1 first.
- AgentCore runtime permissions configured

### Understanding AgentCore Memory

AgentCore Memory provides two types of memory for your agents:

**Short-Term Memory (Conversation Continuity)**
- Maintains context within a single conversation session
- Remembers what was discussed earlier in the conversation
- Enables natural multi-turn interactions
- Implemented through **SummaryStrategy** that creates conversation summaries

**Long-Term Memory (User Preferences)**
- Learns user preferences across multiple sessions
- Stores personalization data using RAG-based retrieval
- Enables the agent to remember user-specific information over time
- Implemented through **UserPreferenceStrategy** that extracts and stores preferences

Without memory, each agent invocation is stateless - it has no knowledge of previous interactions. With memory, your agent can maintain context and provide personalized experiences.

Import required libraries and initialize the AgentCore Runtime client to manage memory-enabled agent deployment:

In [1]:
import json
import os
import random, string
import time

import boto3
from bedrock_agentcore_starter_toolkit import Runtime

boto_session = boto3.Session()
region = boto_session.region_name

print(f"current region: {region}")
account_id = boto_session.client("sts").get_caller_identity()["Account"]
print(f"current account: {account_id}")

agentcore_runtime = Runtime()

current region: us-west-2
current account: 298894475115


### Step 1: Create Memory Resource with Strategies

We'll create a memory resource with two strategies:

**SummaryStrategy:**
- Automatically summarizes conversation history
- Stores summaries in namespaces organized by actor (user) and session
- Helps the agent understand conversation context without processing entire history

**UserPreferenceStrategy:**
- Extracts and stores user preferences from conversations
- Organizes preferences by actor ID for personalization
- Enables the agent to remember user-specific information across sessions

The MemoryManager handles the lifecycle of these memory resources and their associated strategies.

In [2]:
from bedrock_agentcore.memory.session import MemorySessionManager
from bedrock_agentcore_starter_toolkit.operations.memory.manager import MemoryManager
from bedrock_agentcore_starter_toolkit.operations.memory.models.strategies import SummaryStrategy, UserPreferenceStrategy

memory_manager = MemoryManager(region_name=region)

memory_response = memory_manager.get_or_create_memory(
    name="intelligent_rag_memory",
    strategies=[
        SummaryStrategy(
            name="SessionSummarizer",
            namespaces=[
                "/summaries/{actorId}/{sessionId}"
            ]
        ),
        UserPreferenceStrategy(
            name="UserPreferencesLearner",
            namespaces=["/users/{actorId}/preferences"]
        )
    ]
)

memory_id = memory_response.get('id')
print(f"Created memory with ID: {memory_id}")

✅ MemoryManager initialized for region: us-west-2


Memory already exists. Using existing memory ID: intelligent_rag_memory-8LHCm9H2Jb
🔎 Retrieving memory resource with ID: intelligent_rag_memory-8LHCm9H2Jb...
  Found memory: intelligent_rag_memory-8LHCm9H2Jb
Existing {'type': 'SUMMARIZATION', 'name': 'SessionSummarizer', 'description': None, 'namespaces': ['/summaries/{actorId}/{sessionId}']}
Requested {'type': 'SUMMARIZATION', 'name': 'SessionSummarizer', 'description': None, 'namespaces': ['/summaries/{actorId}/{sessionId}']}
Existing {'type': 'USER_PREFERENCE', 'name': 'UserPreferencesLearner', 'description': None, 'namespaces': ['/users/{actorId}/preferences']}
Requested {'type': 'USER_PREFERENCE', 'name': 'UserPreferencesLearner', 'description': None, 'namespaces': ['/users/{actorId}/preferences']}
Universal strategy validation passed for memory intelligent_rag_memory. Strategies match: [SUMMARIZATION, USER_PREFERENCE]


Created memory with ID: intelligent_rag_memory-8LHCm9H2Jb


### Step 2: Store Memory ID for Runtime Access

The memory ID needs to be accessible to the agent runtime when it's deployed to AgentCore. We store it in SSM Parameter Store because:

- **Runtime Access**: AgentCore can retrieve the ID at startup without hardcoding
- **Security**: SSM provides secure parameter storage with IAM-based access control
- **Flexibility**: Easy to update the memory ID without redeploying the agent
- **Consistency**: Same pattern used for knowledge base IDs

The agent code will retrieve this parameter when initializing the memory client.

In [3]:
param_name = '/app/intelligent_rag/agentcore/memory_id'

ssm = boto3.client("ssm")
ssm.put_parameter(Name=param_name, Value=memory_id, Type="String", Overwrite=True)
print(f"Stored {memory_id} in SSM: {param_name}")

Stored intelligent_rag_memory-8LHCm9H2Jb in SSM: /app/intelligent_rag/agentcore/memory_id


### Step 3: Understand the Memory Integration Approach

The `intelligent_rag_agent_runtime_with_memory.py` uses **AgentCoreMemorySessionManager** for native memory integration:

**Approach:**
- Uses `AgentCoreMemorySessionManager` from Strands Agents SDK
- Automatic conversation persistence
- Built-in retrieval from memory namespaces
- Configurable retrieval per namespace (top_k, relevance_score)

**Key Components:**
1. **AgentCoreMemoryConfig**: Configures memory ID, session ID, actor ID, and retrieval settings
2. **RetrievalConfig**: Defines how to retrieve from each memory namespace (summaries, preferences)
3. **AgentCoreMemorySessionManager**: Handles automatic memory persistence and retrieval

**Memory Namespaces:**
- `/summaries/{actorId}/{sessionId}`: Short-term conversation summaries
- `/users/{actorId}/preferences`: Long-term user preferences

Get the notebook directory path to locate the memory-enabled agent code and updated IAM policy:

In [4]:
import os
from pathlib import Path

import ipynbname

try:
    # Get the notebook name and path
    notebook_path = ipynbname.path()
    notebook_dir = Path(notebook_path.parent)
except:
    notebook_dir = Path.resolve()

print(f"notebook_dir: {notebook_dir}")

# Agentcore starter toolkit expects the files in the current directory
# So changing our working directory.
os.chdir(notebook_dir)
print(f"changed working directory to: {notebook_dir}")

notebook_dir: /Workshop/Lab 3 - AgentCore Deployment and Memory
changed working directory to: /Workshop/Lab 3 - AgentCore Deployment and Memory


Load the IAM policy with memory permissions and create/update the execution role for the memory-enabled agent:

In [5]:
# Load policy from external file
with open('policy.json', 'r') as f:
    policy_template = f.read()

# load trust policy from external file
with open('trust-policy.json', 'r') as f:
    trust_policy_template = f.read()

policy = policy_template.replace('REGION', region).replace('ACCOUNT_ID', account_id).replace('REPO_ARN', '*')
print(f"Policy loaded and updated with region: {region}, account: {account_id}")

trust_policy = trust_policy_template.replace('REGION', region).replace('ACCOUNT_ID', account_id)

# create IAM role using the policies
suffix = random.choices(string.ascii_lowercase + string.digits, k=8)
iam_client = boto3.client('iam')
role_name = f"bedrock-runtime-execution-role-{''.join(suffix)}"

role = iam_client.create_role(
    RoleName=role_name,
    AssumeRolePolicyDocument=trust_policy
)

iam_client.put_role_policy(
    RoleName=role_name,
    PolicyName='bedrock-runtime-execution-policy',
    PolicyDocument=policy
)

Policy loaded and updated with region: us-west-2, account: 298894475115


{'ResponseMetadata': {'RequestId': 'cf9598e0-7d7f-4539-9e1e-da7e8d0e2c12',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Fri, 13 Mar 2026 00:00:43 GMT',
   'x-amzn-requestid': 'cf9598e0-7d7f-4539-9e1e-da7e8d0e2c12',
   'content-type': 'text/xml',
   'content-length': '206'},
  'RetryAttempts': 0}}

### Step 4: Deploy Updated Agent with Memory

The memory-enabled agent uses a different entrypoint file: `intelligent_rag_agent_runtime_with_memory.py`

**Key Differences:**
- Initializes MemorySessionManager for conversation management
- Registers MemoryHookProvider to capture conversation events
- Retrieves memory ID from SSM Parameter Store
- Configures actor and session IDs for proper memory organization

The deployment process is identical to the base agent, but the runtime behavior now includes automatic memory management.

Trigger the deployment of the memory-enabled agent to AgentCore Runtime:

In [6]:
agent_name = "intelligent_rag_agent_with_memory"
response = agentcore_runtime.configure(
    entrypoint="intelligent_rag_agent_runtime_with_memory.py",
    execution_role=role['Role']['Arn'],
    #auto_create_execution_role=True,   # If you don't pass in an execution role, AgentCore can create a minimal execution role for you.
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name
    # memory_mode="STM_AND_LTM"   # NOTE:  We created the memory above. You can also have AgentCore Toolkit create the memory for you.
)
response

Entrypoint parsed: file=/Workshop/Lab 3 - AgentCore Deployment and Memory/intelligent_rag_agent_runtime_with_memory.py, bedrock_agentcore_name=intelligent_rag_agent_runtime_with_memory
Memory disabled - agent will be stateless
Configuring BedrockAgentCore agent: intelligent_rag_agent_with_memory
Memory disabled
Network mode: PUBLIC
Generated Dockerfile: Dockerfile
Generated .dockerignore: /Workshop/Lab 3 - AgentCore Deployment and Memory/.dockerignore
Changing default agent from 'intelligent_rag_agent' to 'intelligent_rag_agent_with_memory'
Bedrock AgentCore configured: /Workshop/Lab 3 - AgentCore Deployment and Memory/.bedrock_agentcore.yaml


ConfigureResult(config_path=PosixPath('/Workshop/Lab 3 - AgentCore Deployment and Memory/.bedrock_agentcore.yaml'), dockerfile_path=PosixPath('/Workshop/Lab 3 - AgentCore Deployment and Memory/Dockerfile'), dockerignore_path=PosixPath('/Workshop/Lab 3 - AgentCore Deployment and Memory/.dockerignore'), runtime='Docker', runtime_type=None, region='us-west-2', account_id='298894475115', execution_role='arn:aws:iam::298894475115:role/bedrock-runtime-execution-role-zimoeb4i', ecr_repository=None, auto_create_ecr=True, s3_path=None, auto_create_s3=False, memory_id=None, network_mode='PUBLIC', network_subnets=None, network_security_groups=None, network_vpc_id=None)

Monitor deployment status and store the agent ARN for use in subsequent notebooks:

In [7]:
launch_result = agentcore_runtime.launch(auto_update_on_conflict=True)

🚀 Launching Bedrock AgentCore (cloud mode - RECOMMENDED)...
   • Deploy Python code directly to runtime
   • No Docker required (DEFAULT behavior)
   • Production-ready deployment

💡 Deployment options:
   • runtime.launch()                → Cloud (current)
   • runtime.launch(local=True)      → Local development
Memory disabled - skipping memory creation
Starting CodeBuild ARM64 deployment for agent 'intelligent_rag_agent_with_memory' to account 298894475115 (us-west-2)
Setting up AWS resources (ECR repository, execution roles)...
Getting or creating ECR repository for agent: intelligent_rag_agent_with_memory
ECR repository available: 298894475115.dkr.ecr.us-west-2.amazonaws.com/bedrock-agentcore-intelligent_rag_agent_with_memory
Using execution role from config: arn:aws:iam::298894475115:role/bedrock-runtime-execution-role-zimoeb4i
Preparing CodeBuild project and uploading source...


✅ Reusing existing ECR repository: 298894475115.dkr.ecr.us-west-2.amazonaws.com/bedrock-agentcore-intelligent_rag_agent_with_memory


Getting or creating CodeBuild execution role for agent: intelligent_rag_agent_with_memory
Role name: AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-e2c35e1d32
Reusing existing CodeBuild execution role: arn:aws:iam::298894475115:role/AmazonBedrockAgentCoreSDKCodeBuild-us-west-2-e2c35e1d32
Using dockerignore.template with 43 patterns for zip filtering
Uploaded source to S3: intelligent_rag_agent_with_memory/source.zip
Updated CodeBuild project: bedrock-agentcore-intelligent_rag_agent_with_memory-builder
Starting CodeBuild build (this may take several minutes)...
Starting CodeBuild monitoring...
🔄 QUEUED started (total: 0s)
✅ QUEUED completed in 1.0s
🔄 PROVISIONING started (total: 1s)
✅ PROVISIONING completed in 10.3s
🔄 DOWNLOAD_SOURCE started (total: 11s)
✅ DOWNLOAD_SOURCE completed in 1.0s
🔄 BUILD started (total: 12s)
✅ BUILD completed in 18.6s
🔄 POST_BUILD started (total: 31s)
✅ POST_BUILD completed in 10.3s
🔄 FINALIZING started (total: 41s)
✅ FINALIZING completed in 1.0s
🔄 COMPLETED sta

In [8]:
import time
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    print(status)
print(status)

# Store the agent ARN for use in other notebooks
if status == 'READY':
    try:
        agent_arn = status_response.endpoint['agentRuntimeArn']
        %store agent_arn
        print(f"Agent ARN stored for use in other notebooks: {agent_arn}")
    except Exception as e:
        print(f"Could not store agent ARN: {e}")

Retrieved Bedrock AgentCore status for: intelligent_rag_agent_with_memory


READY
Stored 'agent_arn' (str)
Agent ARN stored for use in other notebooks: arn:aws:bedrock-agentcore:us-west-2:298894475115:runtime/intelligent_rag_agent_with_memory-u1u0EKCsy1


### Step 5: Test Conversation Continuity

When invoking the memory-enabled agent, the **session_id** parameter is crucial:

**Same Session ID:**
- Agent maintains conversation context
- Can reference previous messages
- Builds on earlier discussion

**Different Session ID:**
- Starts a fresh conversation
- No access to previous session context
- Useful for testing or new conversation threads

**Actor ID:**
- Identifies the user across sessions
- Enables long-term preference learning
- Allows personalization based on user history

**Note on Query Performance:** Queries may take 2+ minutes to complete as the agent makes 10+ tool calls to route between knowledge bases, retrieve data, and generate responses. Optimizing agentic systems often involves reducing the number of LLM generation calls and the size of LLM outputs. As you add tools and features, you may need to review agent trajectories and optimize performance. In Lab 3.3, you'll learn how to use AgentCore Observability to analyze these patterns and identify optimization opportunities.


Create a convenience function that includes session tracking to enable memory persistence across invocations:

In [9]:
import json

def print_response_text(invoke_response):
    response = invoke_response['response']
    
    # If it's a list, join all parts first
    if isinstance(response, list):
        response = ''.join(response)
    
    # Parse JSON
    response = json.loads(response)
    
    # Extract the text content
    text = response['result']['content'][0]['text']
    print(text)

In [10]:
def ask(question, session):
    print(f"Query: {question}")
    print("-" * 100)
    response = agentcore_runtime.invoke(
        payload={"prompt": question},
        session_id=session)
    print_response_text(response)
    print("\n")

In [11]:
ask(
    question="How many customers reviewed product_890, are those reviews positive or negative?", 
    session="session111111111111111111111111111")

Query: How many customers reviewed product_890, are those reviews positive or negative?
----------------------------------------------------------------------------------------------------
Based on the context from your previous lookup, here's the summary for **product_890**:

**Number of Customers:** 4 customers reviewed product_890

**Sentiment:** The reviews are **predominantly negative** with an average rating of **1.75 out of 5 stars**.

### Rating Breakdown:
- **2 reviews with 1 star (Very Negative)** - Customers reported critical failures like warped plastic, broken handles, and cracked bowls
- **1 review with 2 stars (Negative)** - Product works but has significant issues, not recommended
- **1 review with 3 stars (Mixed)** - Adequate for basic tasks but disappointed with quality and durability concerns

### Main Complaints:
- Poor durability (failures within 1-2 uses)
- Cheap materials that warp and crack easily
- Flimsy handles that break
- Inadequate non-slip grip
- Overpric

In [12]:
# Try explicitly stopping the session and asking a question about previous conversations

# To stop a session:
agentcore_runtime.stop_session(session_id="session111111111111111111111111111")

Stopping session for agent: intelligent_rag_agent_with_memory
Using provided session ID: session111111111111111111111111111
Session stopped successfully: session111111111111111111111111111
Cleared session ID from configuration
Session stopped: session111111111111111111111111111


{'session_id': 'session111111111111111111111111111',
 'agent_name': 'intelligent_rag_agent_with_memory',
 'status_code': 200,
 'message': 'Session stopped successfully'}

In [13]:
ask(
    question="What review did I just lookup?", 
    session="session111111111111111111111111111")

Query: What review did I just lookup?
----------------------------------------------------------------------------------------------------
You just looked up reviews for **product_890**, a **mixing bowl set** (kitchenware product).

The lookup showed that:
- **4 customers** reviewed this product
- The reviews were **predominantly negative** with an average rating of **1.75 out of 5 stars**
- Main issues included poor quality, durability problems (warping, cracking, broken handles), flimsy design, and complaints about overpricing

The 4 specific reviews ranged from 1-3 stars, with 2 very negative 1-star reviews, 1 negative 2-star review, and 1 mixed 3-star review.




### Test Memory Continuity and User Preferences

The agent now has memory capabilities for conversation continuity and user preferences. Let's test these features:"""

* Try asking the same question with a different session ID
* Try asking the same session ID about previous questions

In [15]:
#  ask questions with a different session ID
ask(
   question="How many customers purhcased product_890?",
   session="session222222222222222222222222222222")
#
ask(
   question="What review did I just lookup?",
   session="session111111111111111111111111111")

# # To stop a session:
# agentcore_runtime.stop_session(session_id="xxxxx")

Query: How many customers purhcased product_890?
----------------------------------------------------------------------------------------------------
Based on the previous query I ran, **0 customers purchased product_890**.

The structured data analysis showed that there are no purchase records for product_890 in the database. This means either:
- The product hasn't been ordered by any customers yet, or
- The product doesn't exist in the order records

Is there another product you'd like me to check, or would you like more details about product_890?


Query: What review did I just lookup?
----------------------------------------------------------------------------------------------------
You just looked up reviews for **product_890**, a **mixing bowl set** (kitchenware product).

The lookup revealed:
- **4 customers** reviewed this product
- Reviews were **predominantly negative** with an average rating of **1.75 out of 5 stars**
- **Rating distribution:**
  - 2 reviews with 1 star (ve

## Next Steps

Your agent now has memory capabilities for conversation continuity and user preferences!

**Ready to continue?** Proceed to [**Lab 3.3**](3.3-agentcore-observability.ipynb) to explore AgentCore Observability features including CloudWatch dashboards, traces, and transaction search.
